# Notebook 5: Exploración y estructuración de los puntos de los puertos

## OceanWatch Analytics

### Presentado por: Gupo 2

#### Pablo Daniel Díaz Mera
#### Ana Catalina Gelvez
#### Christian Rene Mendez

## 0. Instalación de librerías requeridas

In [0]:
%pip install geopandas pyogrio
%pip install folium

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Definición de la capa a evaluar

In [0]:

import os
import zipfile

# Reemplazar con la ruta real del archivo
ruta_zip = "/Volumes/workspace/oceanwatch/geo_data/Raw/WPI_Shapefile.zip"

print("Archivo encontrado:", os.path.isfile(ruta_zip))
print("Tamaño ZIP (MB):", round(os.path.getsize(ruta_zip) / 1024**2, 2))


Archivo encontrado: True
Tamaño ZIP (MB): 1.1


In [0]:
with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    archivos = zip_ref.infolist()

    for archivo in archivos:
        print(
            f"{archivo.filename} | "
            f"Tamaño: {archivo.file_size / 1024**2:.2f} MB"
        )


WPI.cpg | Tamaño: 0.00 MB
WPI.dbf | Tamaño: 51.29 MB
WPI.prj | Tamaño: 0.00 MB
WPI.sbn | Tamaño: 0.03 MB
WPI.sbx | Tamaño: 0.00 MB
WPI.shp | Tamaño: 0.10 MB
WPI.shx | Tamaño: 0.03 MB
WPI2019.xls | Tamaño: 2.43 MB


## 2. Exploración de la capa

### Información general de la capa

In [0]:
import geopandas as gpd

# Leer directamente el Shapefile comprimido
gdf_puertos = gpd.read_file(
    f"/vsizip/{ruta_zip}/WPI.shp",
    engine="pyogrio"
)

print("Total de registros:", len(gdf_puertos))
print("Total de columnas:", len(gdf_puertos.columns))
print("Sistema de referencia:", gdf_puertos.crs)
print("Tipo de geometría:")
print(gdf_puertos.geom_type.value_counts())


Total de registros: 3630
Total de columnas: 79
Sistema de referencia: EPSG:4326
Tipo de geometría:
Point    3630
Name: count, dtype: int64


### Identificación de campos

In [0]:
import pandas as pd

# Estructura de los campos
estructura = pd.DataFrame({
    "Campo": gdf_puertos.columns,
    "Tipo_Dato": gdf_puertos.dtypes.astype(str).values,
    "Nulos": gdf_puertos.isna().sum().values,
    "Valores_Unicos": [
        gdf_puertos[col].nunique(dropna=True)
        for col in gdf_puertos.columns
    ]
})

display(estructura)


Campo,Tipo_Dato,Nulos,Valores_Unicos
INDEX_NO,float64,0,3630
REGION_NO,float64,0,347
PORT_NAME,object,0,3540
COUNTRY,object,0,195
LATITUDE,float64,0,2458
LONGITUDE,float64,0,3006
LAT_DEG,float64,0,78
LAT_MIN,float64,0,60
LAT_HEMI,object,1,2
LONG_DEG,float64,0,180


- Se encuentra que la capa de puntos asociada al World Port Index contiene el detalle del país al cual pertenece el puerto (cammpo "COUNTRY"). Este campo, permitiría hacer una selección de los puertos de Estados Unidos para trabajar la información asociada a ellos.

### Visualización espacial de los puntos del World Port Index 

In [0]:

import folium

# Crear mapa base
mapa_puertos = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="OpenStreetMap",
    prefer_canvas=True
)

# Agregar cada puerto como un punto
for _, puerto in gdf_puertos.iterrows():

    if puerto.geometry is None or puerto.geometry.is_empty:
        continue

    folium.CircleMarker(
        location=[
            puerto.geometry.y,
            puerto.geometry.x
        ],
        radius=3,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        weight=1,
        tooltip=str(puerto["PORT_NAME"]),
        popup=folium.Popup(
            f"Puerto: {puerto['PORT_NAME']}<br>"
            f"País: {puerto['COUNTRY']}",
            max_width=250
        )
    ).add_to(mapa_puertos)

# Mostrar mapa
displayHTML(mapa_puertos._repr_html_())


Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_7282c183209c11c5d0baefff33efa324 {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
</head>
<body>
 
 
 <div class="folium-map" id="map_7282c183209c11c5d0baefff33efa324" ></div>
 
</body>
<script>
 
 
 var map_7282c183209c11c5d0baefff33efa324 = L.map(
 "map_7282c183209c11c5d0baefff33efa324",
 {
 center: [20.0, 0.0],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 2,
 "zoomControl": true,
 "preferCanvas": true,
}

 }
 );

 

 
 
 var tile_layer_638075c91f7e4e52daa337078bde3350 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_638075c91f7e4e52daa337078bde3350.addTo(map_7282c183209c11c5d0baefff33efa324);
 
 
 var circle_marker_ed2b3a4081682221ce2f4f8ba0c56b2a = L.circleMarker(
 [43.86666666666667, 146.83333333333334],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.7, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 3, "stroke": true, "weight": 1}
 ).addTo(map_7282c183209c11c5d0baefff33efa324);
 
 
 var popup_c65a89d1d13c629ba078228dfbabc480 = L.popup({
 "maxWidth": 250,
});

 
 
 var html_23e558dafb30651a4e86003e9b4abd17 = $(`<div id="html_23e558dafb30651a4e86003e9b4abd17" style="width: 100.0%; height: 100.0%;">Puerto: SHAKOTAN<br>País: RU</div>`)[0];
 popup_c65a89d1d13c629ba078228dfbabc480.setContent(html_23e558dafb30651a4e86003e9b4abd17);
 
 

 circle_marker_ed2b3a4081682221ce2f4f8ba0c56b2a.bindPopup(popup_c65a89d1d13c629ba078228dfbabc480)
 ;

 
 
 
 circle_marker_ed2b3a4081682221ce2f4f8ba0c56b2a.bindTooltip(
 `<div>
 SHAKOTAN
 </div>`,
 {
 "sticky": true,
}
 );
 
 
 var circle_marker_50a3310d482790600b364ab237418eb2 = L.circleMarker(
 [44.35, 143.35],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.7, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 3, "stroke": true, "weight": 1}
 ).addTo

## 3. Selección de los puertos de Estados Unidos

### Valores únicos del campo de país

In [0]:
# Obtener valores únicos del campo COUNTRY
paises_unicos = sorted(
    gdf_puertos["COUNTRY"].dropna().unique()
)

print(f"Total de países únicos: {len(paises_unicos)}")
print("\nListado de países:")

for pais in paises_unicos:
    print(pais)


Total de países únicos: 195

Listado de países:
AE
AG
AL
AO
AQ
AR
AS
AU
AW
BA
BB
BD
BE
BG
BH
BJ
BM
BN
BQ
BR
BS
BZ
CA
CD
CG
CI
CK
CL
CM
CN
CO
CR
CU
CV
CW
CX
CY
DE
DJ
DK
DM
DO
DZ
EC
EE
EG
EH
ER
ES
FI
FJ
FK
FM
FO
FR
GA
GB
GD
GE
GF
GG
GH
GI
GL
GM
GN
GP
GQ
GR
GS
GT
GU
GW
GY
HK
HN
HR
HT
ID
IE
IL
IM
IN
IO
IQ
IR
IS
IT
JE
JM
JO
JP
KE
KH
KI
KM
KN
KP
KR
KW
KY
LB
LC
LK
LR
LT
LV
LY
MA
MC
ME
MG
MH
MM
MO
MP
MQ
MR
MT
MU
MV
MX
MY
MZ
NA
NC
NF
NG
NI
NL
NO
NR
NU
NZ
OM
PA
PE
PF
PG
PH
PK
PL
PM
PR
PT
PW
PY
QA
QM
QW
RE
RO
RU
SA
SB
SC
SD
SE
SG
SH
SI
SL
SN
SO
SR
ST
SV
SX
SY
TC
TG
TH
TL
TN
TO
TR
TT
TV
TW
TZ
UA
US
UY
VC
VE
VG
VI
VN
VU
WF
WS
XR
XU
YE
ZA


### Selección de los puertos de Estados Unidos ("COUNTRY"="US")

In [0]:
# Seleccionar puertos de Estados Unidos
gdf_puertos_usa = gdf_puertos.loc[
    gdf_puertos["COUNTRY"] == "US"
].copy()

# Validar selección
print("Total de puertos originales:", len(gdf_puertos))
print("Puertos de Estados Unidos:", len(gdf_puertos_usa))
print("Columnas:", len(gdf_puertos_usa.columns))
print("CRS:", gdf_puertos_usa.crs)
print("Geometrías válidas:", gdf_puertos_usa.geometry.is_valid.sum())

# Visualizar primeros registros
display(
    gdf_puertos_usa[
        ["INDEX_NO", "PORT_NAME", "COUNTRY", "geometry"]
    ].head(10)
)

Total de puertos originales: 3630
Puertos de Estados Unidos: 666
Columnas: 79
CRS: EPSG:4326
Geometrías válidas: 666


,INDEX_NO,PORT_NAME,COUNTRY,geometry
50,4800.0,CHICAGO,US,POINT (-87.6 41.88333)
51,4810.0,WILMETTE,US,POINT (-87.68333 42.08333)
52,4820.0,GREAT LAKES,US,POINT (-87.83333 42.3)
53,4830.0,WAUKEGAN,US,POINT (-87.81667 42.36667)
54,4840.0,KENOSHA,US,POINT (-87.81667 42.58333)
55,4850.0,RACINE,US,POINT (-87.78333 42.73333)
56,4860.0,MILWAUKEE,US,POINT (-87.88333 43.03333)
57,4870.0,PORT WASHINGTON,US,POINT (-87.86667 43.38333)
58,4880.0,SHEBOYGAN,US,POINT (-87.7 43.75)
59,4890.0,MANITOWOC,US,POINT (-87.65 44.1)


- Se encuentra que 666 puertos corresponden a Estados Unidos lo cual representa el 18.3% del total de puertos

### Visualización de los puertos de Estados Unidos (seleccionados por tabla de atributos)

In [0]:
# Crear mapa base centrado en Estados Unidos
mapa_puertos_usa = folium.Map(
    location=[39, -98],
    zoom_start=4,
    tiles="OpenStreetMap",
    prefer_canvas=True
)

# Agregar cada puerto como un punto individual
for _, puerto in gdf_puertos_usa.iterrows():

    if puerto.geometry is None or puerto.geometry.is_empty:
        continue

    folium.CircleMarker(
        location=[
            puerto.geometry.y,
            puerto.geometry.x
        ],
        radius=3,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        weight=1,
        tooltip=str(puerto["PORT_NAME"]),
        popup=folium.Popup(
            f"Puerto: {puerto['PORT_NAME']}<br>"
            f"País: {puerto['COUNTRY']}",
            max_width=250
        )
    ).add_to(mapa_puertos_usa)

# Mostrar mapa interactivo en Databricks
displayHTML(mapa_puertos_usa._repr_html_())


Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_0dddd77e693341150e5fca6e5b4d3b93 {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
</head>
<body>
 
 
 <div class="folium-map" id="map_0dddd77e693341150e5fca6e5b4d3b93" ></div>
 
</body>
<script>
 
 
 var map_0dddd77e693341150e5fca6e5b4d3b93 = L.map(
 "map_0dddd77e693341150e5fca6e5b4d3b93",
 {
 center: [39.0, -98.0],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 4,
 "zoomControl": true,
 "preferCanvas": true,
}

 }
 );

 

 
 
 var tile_layer_762ae667ca1af964cac4212e05901c0e = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_762ae667ca1af964cac4212e05901c0e.addTo(map_0dddd77e693341150e5fca6e5b4d3b93);
 
 
 var circle_marker_1dbd3dec3a81f3b9a54e6cbf8ff701d0 = L.circleMarker(
 [41.88333333333333, -87.6],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.7, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 3, "stroke": true, "weight": 1}
 ).addTo(map_0dddd77e693341150e5fca6e5b4d3b93);
 
 
 var popup_2f466150657f178a2e18f08152730107 = L.popup({
 "maxWidth": 250,
});

 
 
 var html_b68f1bac80553317a796797731cc9750 = $(`<div id="html_b68f1bac80553317a796797731cc9750" style="width: 100.0%; height: 100.0%;">Puerto: CHICAGO<br>País: US</div>`)[0];
 popup_2f466150657f178a2e18f08152730107.setContent(html_b68f1bac80553317a796797731cc9750);
 
 

 circle_marker_1dbd3dec3a81f3b9a54e6cbf8ff701d0.bindPopup(popup_2f466150657f178a2e18f08152730107)
 ;

 
 
 
 circle_marker_1dbd3dec3a81f3b9a54e6cbf8ff701d0.bindTooltip(
 `<div>
 CHICAGO
 </div>`,
 {
 "sticky": true,
}
 );
 
 
 var circle_marker_3fc18e572af91443db1720b27fa2371d = L.circleMarker(
 [42.083333333333336, -87.68333333333334],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.7, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 3, "stroke": true, "weight":

### Selección de los puertos de estados unidos (cruce espacial con límites marítimos y terrestres de Estados Unidos)

#### Cara de la capa de límites terrestres y marítimos

In [0]:
import geopandas as gpd

# Ruta de límites terrestres y marítimos
ruta_limites_us = (
    "/Volumes/workspace/oceanwatch/geo_data/Processed/"
    "Limites_Terrestres_Maritimos_US_WGS84.zip"
)

# Cargar capa geográfica
gdf_limites_us = gpd.read_file(ruta_limites_us)

# Verificar información
print("LÍMITES TERRESTRES Y MARÍTIMOS US")
print("-" * 50)
print(f"Registros: {len(gdf_limites_us)}")
print(f"CRS: {gdf_limites_us.crs}")
print(f"Geometría: {gdf_limites_us.geom_type.iloc[0]}")
print(f"Geometría válida: {gdf_limites_us.is_valid.all()}")


LÍMITES TERRESTRES Y MARÍTIMOS US
--------------------------------------------------
Registros: 1
CRS: EPSG:4326
Geometría: MultiPolygon
Geometría válida: True


### Verificación de sistemas de referencia

In [0]:

# Verificar sistemas de referencia
print("SISTEMAS DE REFERENCIA")
print("-" * 50)

print(f"Puertos: {gdf_puertos.crs}")
print(f"Límites US: {gdf_limites_us.crs}")

print(f"\nTotal de puertos: {len(gdf_puertos):,}")
print(f"Total de límites: {len(gdf_limites_us)}")

# Crear copia de los puertos en WGS84
if gdf_puertos.crs is None:
    raise ValueError("La capa de puertos no tiene CRS definido.")

elif gdf_puertos.crs.to_epsg() == 4326:
    gdf_puertos_wgs84 = gdf_puertos.copy()

else:
    gdf_puertos_wgs84 = gdf_puertos.to_crs(
        epsg=4326
    )

print("\nCRS FINAL")
print(gdf_puertos_wgs84.crs)


SISTEMAS DE REFERENCIA
--------------------------------------------------
Puertos: EPSG:4326
Límites US: EPSG:4326

Total de puertos: 3,630
Total de límites: 1

CRS FINAL
EPSG:4326


### Selección espacial de los puertos

In [0]:

# Obtener geometría de límites US
geom_limites_us = gdf_limites_us.geometry.iloc[0]

# Seleccionar puertos que intersectan los límites
gdf_puertos_us_espacial = gdf_puertos_wgs84[
    gdf_puertos_wgs84.geometry.intersects(
        geom_limites_us
    )
].copy()

# Resultados
print("RESULTADOS DE LA SELECCIÓN ESPACIAL")
print("-" * 50)

print(f"Puertos originales: {len(gdf_puertos_wgs84):,}")
print(f"Puertos seleccionados: {len(gdf_puertos_us_espacial):,}")
print(
    f"Puertos fuera de la cobertura: "
    f"{len(gdf_puertos_wgs84) - len(gdf_puertos_us_espacial):,}"
)

print(f"\nCRS: {gdf_puertos_us_espacial.crs}")

# Primeros registros
display(
    gdf_puertos_us_espacial[
        ["PORT_NAME", "COUNTRY", "geometry"]
    ].head(10)
)


RESULTADOS DE LA SELECCIÓN ESPACIAL
--------------------------------------------------
Puertos originales: 3,630
Puertos seleccionados: 641
Puertos fuera de la cobertura: 2,989

CRS: EPSG:4326


,PORT_NAME,COUNTRY,geometry
53,WAUKEGAN,US,POINT (-87.81667 42.36667)
54,KENOSHA,US,POINT (-87.81667 42.58333)
55,RACINE,US,POINT (-87.78333 42.73333)
59,MANITOWOC,US,POINT (-87.65 44.1)
61,BAY CITY,US,POINT (-83.86667 43.6)
194,PORT ALUCROIX,VI,POINT (-64.76667 17.7)
309,POINT SAN PABLO,US,POINT (-122.43333 37.96667)
310,SAUSALITO,US,POINT (-122.48333 37.85)
311,PINOLE POINT,US,POINT (-122.36667 38.01667)
312,OLEUM,US,POINT (-122.26667 38.05)


### Comparación de los puertos seleccionados por las dos metodologías

In [0]:
# Selección por atributos
gdf_puertos_us_atributos = gdf_puertos_wgs84[
    gdf_puertos_wgs84["COUNTRY"] == "US"
].copy()

# Identificar registros por su índice original
idx_atributos = set(gdf_puertos_us_atributos.index)
idx_espacial = set(gdf_puertos_us_espacial.index)

# Coincidencias
idx_coincidentes = idx_atributos & idx_espacial

# US por atributo, fuera de la selección espacial
idx_solo_atributos = idx_atributos - idx_espacial

# Dentro del límite, pero COUNTRY diferente de US
idx_solo_espacial = idx_espacial - idx_atributos

print("COMPARACIÓN DE MÉTODOS")
print("-" * 50)

print(f"Selección por atributos: {len(idx_atributos)}")
print(f"Selección espacial: {len(idx_espacial)}")
print(f"Coinciden en ambos métodos: {len(idx_coincidentes)}")
print(f"Solo por atributos: {len(idx_solo_atributos)}")
print(f"Solo por selección espacial: {len(idx_solo_espacial)}")

# Crear capas para revisar diferencias
gdf_solo_atributos = gdf_puertos_wgs84.loc[
    sorted(idx_solo_atributos)
].copy()

gdf_solo_espacial = gdf_puertos_wgs84.loc[
    sorted(idx_solo_espacial)
].copy()

# Mostrar puertos US fuera de la cobertura
print("\nPUERTOS US FUERA DE LA COBERTURA ESPACIAL")
display(
    gdf_solo_atributos[
        ["PORT_NAME", "COUNTRY", "LATITUDE", "LONGITUDE"]
    ]
)

# Mostrar puertos dentro de la cobertura con otro país
print("\nPUERTOS DENTRO DE LA COBERTURA CON OTRO PAÍS")
display(
    gdf_solo_espacial[
        ["PORT_NAME", "COUNTRY", "LATITUDE", "LONGITUDE"]
    ]
)


COMPARACIÓN DE MÉTODOS
--------------------------------------------------
Selección por atributos: 666
Selección espacial: 641
Coinciden en ambos métodos: 612
Solo por atributos: 54
Solo por selección espacial: 29

PUERTOS US FUERA DE LA COBERTURA ESPACIAL


PORT_NAME,COUNTRY,LATITUDE,LONGITUDE
CHICAGO,US,41.883333,-87.6
WILMETTE,US,42.083333,-87.683333
GREAT LAKES,US,42.3,-87.833333
MILWAUKEE,US,43.033333,-87.883333
PORT WASHINGTON,US,43.383333,-87.866667
SHEBOYGAN,US,43.75,-87.7
PORT HOPE,US,43.95,-82.7
ALGONAC,US,42.616667,-82.516667
ROCHESTER,US,43.266667,-77.6
FAIRPORT,US,41.766667,-81.3



PUERTOS DENTRO DE LA COBERTURA CON OTRO PAÍS


PORT_NAME,COUNTRY,LATITUDE,LONGITUDE
PORT ALUCROIX,VI,17.7,-64.766667
JOHNSTON ATOLL,XU,16.75,-169.516667
WAKE ISLAND,QW,19.283333,166.616667
SAIPAN,MP,15.2,145.683333
TINIAN,MP,14.966667,145.616667
ROTA,MP,14.133333,145.15
APRA HARBOR,GU,13.45,144.616667
PAGO PAGO HARBOR,AS,-14.283333,-170.666667
BROCKVILLE,CA,44.583333,-75.666667
AGUADILLA,PR,18.433333,-67.15


### Visualización de los puertos seleccionados por las dos metodologías

In [0]:

import folium

# Crear GeoDataFrame de puertos coincidentes
gdf_ambos = gdf_puertos_wgs84.loc[
    sorted(idx_coincidentes)
].copy()

# Crear mapa
mapa_comparacion = folium.Map(
    location=[39.5, -98.35],
    zoom_start=3,
    tiles="OpenStreetMap"
)

# Agregar límites de Estados Unidos
folium.GeoJson(
    gdf_limites_us,
    name="Límites terrestres y marítimos US",
    style_function=lambda feature: {
        "fillColor": "#80BDE3",
        "color": "#2878B5",
        "weight": 1,
        "fillOpacity": 0.15
    }
).add_to(mapa_comparacion)

# Definir grupos de puntos
grupos = [
    (
        gdf_ambos,
        "Ambos métodos",
        "#2CA25F"
    ),
    (
        gdf_solo_atributos,
        "Solo por atributos",
        "#E69F00"
    ),
    (
        gdf_solo_espacial,
        "Solo por selección espacial",
        "#8856A7"
    )
]

# Agregar puntos de cada grupo
for gdf, nombre, color in grupos:

    capa = folium.FeatureGroup(
        name=f"{nombre} ({len(gdf)})",
        show=True
    )

    for _, puerto in gdf.iterrows():

        folium.CircleMarker(
            location=[
                puerto.geometry.y,
                puerto.geometry.x
            ],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            weight=1,
            tooltip=(
                f"Puerto: {puerto['PORT_NAME']} | "
                f"País: {puerto['COUNTRY']} | "
                f"Selección: {nombre}"
            )
        ).add_to(capa)

    capa.add_to(mapa_comparacion)

# Agregar control de capas
folium.LayerControl(
    collapsed=False
).add_to(mapa_comparacion)

# Visualizar en Databricks
displayHTML(mapa_comparacion._repr_html_())


Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_eb7ce8a47119353f1bf12f6cd850c59c {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
</head>
<body>
 
 
 <div class="folium-map" id="map_eb7ce8a47119353f1bf12f6cd850c59c" ></div>
 
</body>
<script>
 
 
 var map_eb7ce8a47119353f1bf12f6cd850c59c = L.map(
 "map_eb7ce8a47119353f1bf12f6cd850c59c",
 {
 center: [39.5, -98.35],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 3,
 "zoomControl": true,
 "preferCanvas": false,
}

 }
 );

 

 
 
 var tile_layer_4386ff534bfc0c05e5c010798d2f5f96 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_4386ff534bfc0c05e5c010798d2f5f96.addTo(map_eb7ce8a47119353f1bf12f6cd850c59c);
 
 
 function geo_json_512adfd61272ad1f5ae00002b0d1e4cd_styler(feature) {
 switch(feature.id) {
 default:
 return {"color": "#2878B5", "fillColor": "#80BDE3", "fillOpacity": 0.15, "weight": 1};
 }
 }

 function geo_json_512adfd61272ad1f5ae00002b0d1e4cd_onEachFeature(feature, layer) {

 layer.on({
 });
 };
 var geo_json_512adfd61272ad1f5ae00002b0d1e4cd = L.geoJson(null, {
 onEachFeature: geo_json_512adfd61272ad1f5ae00002b0d1e4cd_onEachFeature,
 
 style: geo_json_512adfd61272ad1f5ae00002b0d1e4cd_styler,
 ...{
}
 });

 function geo_json_512adfd61272ad1f5ae00002b0d1e4cd_add (data) {
 geo_json_512adfd61272ad1f5ae00002b0d1e4cd
 .addData(data);
 }
 geo_json_512adfd61272ad1f5ae00002b0d1e4cd_add({"bbox": [-180.0, -17.554999988164848, 180.0, 74.70884000326376], "features": [{"bbox": [-180.0, -17.554999988164848, 180.0, 74.70884000326376], "geometry": {"coordinates": [[[[-171.64652439999995, -10.282725152793509], [-171.62966200300005, -10.281516048794458], [-171.61279974499996, -10.28030606379533], [-171.59593760599998, -10.279095206796265], [-171.57907560600003, -10.277883468797215], [-171.56221372499994, -10.276670859798049], [-171.54535197399997, -10.275457368799014], [-171.52849035999995, -10.274242995799918], [-171.51162886699996, -10.27302775180088], [-171.494767503, -10.27181163580178], [-171.477906267, -10.27059463880269], [-171.46104516099993, -10.26

### Selección de todos los puertos seleccionados por las dos metodologías

In [0]:

# Unificar índices de ambas selecciones
idx_puertos_final = idx_atributos | idx_espacial

# Crear GeoDataFrame sin repetir registros
gdf_puertos_us_final = gdf_puertos_wgs84.loc[
    sorted(idx_puertos_final)
].copy()

# Reiniciar índice
gdf_puertos_us_final.reset_index(
    drop=True,
    inplace=True
)

# Verificar resultados
print("UNIFICACIÓN DE PUERTOS")
print("-" * 50)

print(f"Selección por atributos: {len(idx_atributos):,}")
print(f"Selección espacial: {len(idx_espacial):,}")
print(f"Coincidencias: {len(idx_coincidentes):,}")

print(f"\nTotal de puertos únicos: {len(gdf_puertos_us_final):,}")

print(f"\nCRS: {gdf_puertos_us_final.crs}")
print(f"Columnas: {len(gdf_puertos_us_final.columns)}")

print("\nTIPOS DE GEOMETRÍA")
print(gdf_puertos_us_final.geom_type.value_counts())

UNIFICACIÓN DE PUERTOS
--------------------------------------------------
Selección por atributos: 666
Selección espacial: 641
Coincidencias: 612

Total de puertos únicos: 695

CRS: EPSG:4326
Columnas: 79

TIPOS DE GEOMETRÍA
Point    695
Name: count, dtype: int64


## 4. Almacenamiento de los puertos de Estados Unidos

### Almacenamiento en formato shape

In [0]:
import os
import shutil
import zipfile
import tempfile
import geopandas as gpd

# Configuración de salida
directorio_salida = (
    "/Volumes/workspace/oceanwatch/geo_data/Processed/"
)

nombre_capa = "Puertos_US_Seleccion_Final"

ruta_zip_final_puertos = os.path.join(
    directorio_salida,
    f"{nombre_capa}.zip"
)

# Crear directorio temporal
with tempfile.TemporaryDirectory() as directorio_temporal:

    # Ruta del Shapefile temporal
    ruta_shape = os.path.join(
        directorio_temporal,
        f"{nombre_capa}.shp"
    )

    # Exportar Shapefile
    gdf_puertos_us_final.to_file(
        ruta_shape,
        driver="ESRI Shapefile",
        encoding="UTF-8"
    )

    # Crear archivo ZIP temporal
    ruta_zip_temporal = os.path.join(
        directorio_temporal,
        f"{nombre_capa}.zip"
    )

    # Comprimir archivos del Shapefile
    with zipfile.ZipFile(
        ruta_zip_temporal,
        "w",
        zipfile.ZIP_DEFLATED
    ) as zipf:

        for archivo in os.listdir(directorio_temporal):
            if archivo.endswith((
                ".shp", ".shx", ".dbf",
                ".prj", ".cpg"
            )):
                zipf.write(
                    os.path.join(directorio_temporal, archivo),
                    arcname=archivo
                )

    # Copiar ZIP a Unity Catalog
    os.makedirs(directorio_salida, exist_ok=True)

    shutil.copyfile(
        ruta_zip_temporal,
        ruta_zip_final_puertos
    )

# Verificar archivo
print("EXPORTACIÓN COMPLETADA")
print("-" * 50)
print(f"Archivo: {ruta_zip_final_puertos}")
print(
    f"Tamaño: "
    f"{os.path.getsize(ruta_zip_final_puertos) / (1024**2):.2f} MB"
)


EXPORTACIÓN COMPLETADA
--------------------------------------------------
Archivo: /Volumes/workspace/oceanwatch/geo_data/Processed/Puertos_US_Seleccion_Final.zip
Tamaño: 0.08 MB


### Verificación del shape almacenado

In [0]:

# Leer archivo exportado
gdf_puertos_verificacion = gpd.read_file(
    ruta_zip_final_puertos
)

print("VALIDACIÓN DEL ARCHIVO PROCESADO")
print("-" * 50)

print(f"Registros: {len(gdf_puertos_verificacion):,}")
print(f"Columnas: {len(gdf_puertos_verificacion.columns)}")
print(f"CRS: {gdf_puertos_verificacion.crs}")

print("\nTIPOS DE GEOMETRÍA")
print(gdf_puertos_verificacion.geom_type.value_counts())

print("\nGEOMETRÍAS VÁLIDAS")
print(gdf_puertos_verificacion.is_valid.value_counts())

# Comparar cantidad de registros
print("\nVALIDACIÓN DE REGISTROS")
print(
    "Cantidad conservada:",
    len(gdf_puertos_verificacion)
    == len(gdf_puertos_us_final)
)

# Comparar geometrías
print(
    "Geometrías conservadas:",
    gdf_puertos_verificacion.geometry.reset_index(drop=True)
    .geom_equals(
        gdf_puertos_us_final.geometry.reset_index(drop=True)
    ).all()
)


VALIDACIÓN DEL ARCHIVO PROCESADO
--------------------------------------------------
Registros: 695
Columnas: 79
CRS: EPSG:4326

TIPOS DE GEOMETRÍA
Point    695
Name: count, dtype: int64

GEOMETRÍAS VÁLIDAS
True    695
Name: count, dtype: int64

VALIDACIÓN DE REGISTROS
Cantidad conservada: True
Geometrías conservadas: True


## 5. Validación del archivio de los puertos de Estados Unidos

In [0]:
# Leer archivo GeoParquet
gdf_validacion = gpd.read_parquet(ruta_salida)

print("Registros:", len(gdf_validacion))
print("Columnas:", len(gdf_validacion.columns))
print("CRS:", gdf_validacion.crs)
print("Geometrías:", gdf_validacion.geom_type.value_counts())

# Comprobar integridad básica
assert len(gdf_validacion) == len(gdf_puertos_usa)
assert gdf_validacion.crs == gdf_puertos_usa.crs
assert (gdf_validacion["COUNTRY"] == "US").all()

print("Validación completada correctamente")



Registros: 666
Columnas: 79
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": 